# Testing the Initial Qiskit Port

## Objective

Before attempting to convert the full variational Lanczos algorithm, the first goal is to verify that the fundamental linear algebra operations used throughout the algorithm produce identical results to the original Qulacs implementation.

Rather than immediately testing on the Anderson Impurity Model Hamiltonian, simple one-qubit examples are used where the expected results are known analytically.

---

# Step 1 – Verify the New Qiskit Module

The first step was to ensure that the newly created `qiskit_vqe.py` module was correctly recognized by Python.

Actions performed:

- Imported the helper functions:
  - `expectation_value()`
  - `transition_amplitude()`
  - `overlap()`
- Verified the module path using to ensure Python was importing the correct file. (Cell block [2])

to verify that all newly added functions were present.

Because Jupyter caches imported modules, the module was reloaded (Cell block [4]) before testing newly added functions.

This confirmed that the notebook was using the latest version of `qiskit_vqe.py`.

In [1]:
from qiskit_port.qiskit_vqe import expectation_value, transition_amplitude, overlap

print("qiskit_vqe helpers imported")

qiskit_vqe helpers imported


In [2]:
import qiskit_port.qiskit_vqe as qv
print(qv.__file__)
print(dir(qv))

/Users/aporter2/Downloads/nlr_qc/qcaob-aim/qiskit_port/qiskit_vqe.py
['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'expectation_value', 'lanczos_cost_function_qiskit', 'np', 'overlap', 'transition_amplitude']


In [3]:
import importlib
import qiskit_port.qiskit_vqe as qv

importlib.reload(qv)

print("lanczos_cost_function_qiskit" in dir(qv))

True


In [4]:
import importlib
import qiskit_port.qiskit_vqe as qv
importlib.reload(qv)

from qiskit_port.qiskit_vqe import lanczos_cost_function_qiskit
print("import works")

import works


# Step 2 – Test the Helper Functions

Three helper functions were implemented:

- `expectation_value()`
- `transition_amplitude()`
- `overlap()`

These replace the equivalent Qulacs functionality:

| Qulacs | Qiskit/NumPy |
|---------|--------------|
| get_expectation_value() | expectation_value() |
| get_transition_amplitude() | transition_amplitude() |
| inner_product() | overlap() |

These helper routines form the mathematical foundation for the remaining variational Lanczos implementation.

In [10]:
import numpy as np

from qulacs import QuantumState
from qulacs.quantum_operator import create_quantum_operator_from_openfermion_text

from qiskit_port.qiskit_vqe import expectation_value

# Same Hamiltonian in both frameworks
qulacs_H = create_quantum_operator_from_openfermion_text("1.0 [Z0]")
Hmat = np.array([[1, 0],
                 [0, -1]], dtype=complex)

for basis in [0, 1]:

    # Qulacs state
    qs = QuantumState(1)
    qs.set_computational_basis(basis)

    # NumPy state
    state = np.zeros(2, dtype=complex)
    state[basis] = 1

    qulacs_result = qulacs_H.get_expectation_value(qs)
    qiskit_result = expectation_value(state, Hmat)

    print(f"|{basis}>")
    print("Qulacs :", qulacs_result)
    print("Qiskit :", qiskit_result)
    print("-"*40)

|0>
Qulacs : (1+0j)
Qiskit : (1+0j)
----------------------------------------
|1>
Qulacs : (-1+0j)
Qiskit : (-1+0j)
----------------------------------------


## Validation Check: Expectation Value

This cell checks that the numerical difference between the Qulacs result and the Qiskit/NumPy result is below the chosen tolerance.

If the assertion passes, then `expectation_value()` is validated for this test case.

In [12]:
tolerance = 1e-12

assert abs(qulacs_result - qiskit_result) < tolerance

print("expectation_value() matches Qulacs.")

expectation_value() matches Qulacs.


## Qulacs vs Qiskit Test: Inner Product / Overlap

This test compares the original Qulacs inner-product calculation with the new Qiskit/NumPy `overlap()` helper function.

Original Qulacs operation:

`inner_product(qs0, qs1)`

Qiskit replacement:

`overlap(state0, state1)`

The goal is to confirm that Qiskit/NumPy computes overlaps consistently with Qulacs.

In [19]:
from qulacs.state import inner_product
from qiskit_port.qiskit_vqe import overlap

qs0 = QuantumState(1)
qs0.set_computational_basis(0)

qs1 = QuantumState(1)
qs1.set_computational_basis(1)

state0 = np.array([1, 0], dtype=complex)
state1 = np.array([0, 1], dtype=complex)

qulacs_result = inner_product(qs0, qs1)
qiskit_result = overlap(state0, state1)

print("Qulacs overlap:", qulacs_result)
print("Qiskit overlap:", qiskit_result)

Qulacs overlap: 0j
Qiskit overlap: 0j


## Qulacs vs Qiskit Test: Transition Amplitude

This test compares the original Qulacs transition-amplitude calculation with the new Qiskit/NumPy `transition_amplitude()` helper function.

Original Qulacs operation:

`qulacs_H.get_transition_amplitude(qs0, qs1)`

Qiskit replacement:

`transition_amplitude(state0, Hmat, state1)`

This is important because the variational Lanczos cost function depends directly on transition amplitudes between Krylov vectors.

In [15]:
from qiskit_port.qiskit_vqe import transition_amplitude

# Hamiltonian H = X0
qulacs_H = create_quantum_operator_from_openfermion_text("1.0 [X0]")
Hmat = np.array([[0, 1], [1, 0]], dtype=complex)

qs0 = QuantumState(1)
qs0.set_computational_basis(0)

qs1 = QuantumState(1)
qs1.set_computational_basis(1)

state0 = np.array([1, 0], dtype=complex)
state1 = np.array([0, 1], dtype=complex)

# Qulacs equivalent of <0|H|1>
qulacs_result = qulacs_H.get_transition_amplitude(qs0, qs1)
qiskit_result = transition_amplitude(state0, Hmat, state1)

print("Qulacs transition:", qulacs_result)
print("Qiskit transition:", qiskit_result)

Qulacs transition: (1+0j)
Qiskit transition: (1+0j)
Difference: 0.0


## Next Validation Target: Lanczos Cost Function

The helper functions have now been tested against their Qulacs counterparts:

- `expectation_value()`
- `overlap()`
- `transition_amplitude()`

The next function to validate is:

`lanczos_cost_function_qiskit()`

This comparison requires the actual repository ansatz and AIM graph structure, because the original Qulacs function builds the trial Krylov state using `SymmQulacsVqeEmulator`.

Next step:

Use a small real AIM system, such as `N = 2`, and compare:

`vqe.lanczos_cost_function()`

against

`qiskit_port.qiskit_vqe.lanczos_cost_function_qiskit()`

using the same Hamiltonian, same Krylov states, same parameters, and same ansatz structure.

In [16]:
from qiskit import QuantumCircuit

def simple_ansatz_builder(params, n_layers, initial_occupations_indices, connected_graphs):
    qc = QuantumCircuit(1)

    for q in initial_occupations_indices:
        qc.x(q)

    qc.ry(params[0], 0)

    return qc

## Function Equivalence Test: continued_fraction()

This test compares the original repository function `vqe.continued_fraction()` against the Qiskit-port version `continued_fraction_qiskit()`.

The same frequency array `w` and Lanczos coefficient arrays `a` and `b` are passed into both functions.

This validates that the continued-fraction recursion was ported correctly.

This test does not yet validate the generation of Lanczos coefficients. We still need to implement the anderson_impurity_model.py


In [26]:
import numpy as np
import importlib

import vqe
import qiskit_port.qiskit_vqe as qv

importlib.reload(qv)

w = np.linspace(-5, 5, 20, dtype=np.complex128) + 0.1j
a = np.array([1.0, 2.0, 3.0], dtype=np.complex128)
b = np.array([0.0, 0.5, 0.25], dtype=np.complex128)

original = vqe.continued_fraction(w, a, b)
converted = qv.continued_fraction_qiskit(w, a, b)

diff = np.max(np.abs(original - converted))

print("max difference:", diff)

assert np.allclose(original, converted)

print("continued_fraction_qiskit() matches original vqe.continued_fraction()")

max difference: 0.0
continued_fraction_qiskit() matches original vqe.continued_fraction()


In [27]:
git add qiskit_port/qiskit_vqe.py qiskit_conversion_testing.ipynb
git commit -m "Add continued fraction validation test"
git push

SyntaxError: invalid syntax (631966942.py, line 1)